# Lab 07 — Object Detection

**Deep Learning for Computer Vision · AIT**

Classification asks *what is in this image*. Detection asks *what, and where, and how
many* — and that "how many" is what makes it hard: the output length varies per image,
so you cannot just add a softmax and call it a day.

This lab is in three parts, deliberately ordered by how long the ideas will stay useful:

1. **The machinery you will use forever** — box formats, IoU, NMS, mAP. You implement
   these by hand. They are 40 lines and they never go out of date.
2. **The practical path** — fine-tune YOLO11 on your own data with Ultralytics.
   This is what you will actually do in a project.
3. **The idea that replaced anchors** — DETR's *set prediction* with Hungarian matching,
   and the real-time descendants (RT-DETR, YOLOv10) that removed NMS entirely.

> **A note on what we removed.** Earlier versions of this course rebuilt YOLOv3 by
> parsing a Darknet `.cfg` file into PyTorch modules. That taught config parsing more
> than detection, and the architecture is from 2018. The concepts survive here; the
> plumbing does not.

**Runs on:** Colab free T4, ~15 min.

**Reading:** [YOLOv1](https://arxiv.org/abs/1506.02640) ·
[Focal Loss / RetinaNet](https://arxiv.org/abs/1708.02002) ·
[DETR](https://arxiv.org/abs/2005.12872) ·
[RT-DETR](https://arxiv.org/abs/2304.08069) ·
[YOLOv10](https://arxiv.org/abs/2405.14458)

## 0. Setup

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
torch.manual_seed(0); np.random.seed(0)

---
# 1. Boxes, IoU, NMS, mAP

## 1.1 Box formats

Three conventions, and mixing them up is the single most common detection bug:

| Format | Layout | Used by |
|---|---|---|
| `xyxy` | (x1, y1, x2, y2) absolute pixels | torchvision, evaluation code |
| `xywh` | (x, y, w, h), x/y = **top-left** | COCO annotations |
| `cxcywh` | (cx, cy, w, h) **normalised to [0,1]** | YOLO label files, DETR |

Always know which one you are holding. Write the conversions once and use them.

In [ ]:
def xywh_to_xyxy(b):
    x, y, w, h = b.unbind(-1)
    return torch.stack([x, y, x + w, y + h], dim=-1)


def cxcywh_to_xyxy(b):
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)


def xyxy_to_cxcywh(b):
    x1, y1, x2, y2 = b.unbind(-1)
    return torch.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], dim=-1)


b = torch.tensor([[10., 20., 60., 80.]])
print("xyxy   :", b.tolist())
print("cxcywh :", xyxy_to_cxcywh(b).tolist())
print("round  :", cxcywh_to_xyxy(xyxy_to_cxcywh(b)).tolist())

## 1.2 Intersection over Union

IoU is the currency of detection. It decides which prediction matches which ground truth,
which anchors are positive, and whether a detection counts as correct.

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

The whole trick is computing the intersection rectangle with `max` on the top-left
corners and `min` on the bottom-right, then clamping at zero for the non-overlapping case.

In [ ]:
def box_iou(a, b):
    # a: (N,4) xyxy, b: (M,4) xyxy  ->  (N,M) IoU matrix
    area_a = (a[:, 2] - a[:, 0]).clamp(0) * (a[:, 3] - a[:, 1]).clamp(0)
    area_b = (b[:, 2] - b[:, 0]).clamp(0) * (b[:, 3] - b[:, 1]).clamp(0)

    lt = torch.max(a[:, None, :2], b[None, :, :2])       # top-left of intersection
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])       # bottom-right of intersection
    wh = (rb - lt).clamp(min=0)                          # zero if they do not overlap
    inter = wh[..., 0] * wh[..., 1]

    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


A = torch.tensor([[0., 0., 10., 10.], [0., 0., 10., 10.], [0., 0., 10., 10.]])
B = torch.tensor([[0., 0., 10., 10.], [5., 5., 15., 15.], [20., 20., 30., 30.]])
for i, (name, expected) in enumerate([("identical", 1.0), ("half-overlap", 25/175), ("disjoint", 0.0)]):
    print("%-14s IoU = %.4f   (expected %.4f)" % (name, box_iou(A[i:i+1], B[i:i+1]).item(), expected))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for i, name in enumerate(["identical", "half-overlap", "disjoint"]):
    a, bb = A[i], B[i]
    ax[i].add_patch(patches.Rectangle((a[0], a[1]), a[2]-a[0], a[3]-a[1],
                                      fill=False, ec="tab:blue", lw=2, label="A"))
    ax[i].add_patch(patches.Rectangle((bb[0], bb[1]), bb[2]-bb[0], bb[3]-bb[1],
                                      fill=False, ec="tab:red", lw=2, label="B"))
    ax[i].set_xlim(-5, 35); ax[i].set_ylim(-5, 35); ax[i].invert_yaxis()
    ax[i].set_title("%s\nIoU = %.3f" % (name, box_iou(a[None], bb[None]).item()))
    ax[i].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 1.3 Non-Maximum Suppression

A detector fires many times on the same object. NMS keeps the highest-scoring box and
deletes anything overlapping it by more than a threshold, per class.

The algorithm is three lines of logic:
1. Sort by score.
2. Take the top box, keep it.
3. Delete every remaining box whose IoU with it exceeds the threshold. Repeat.

NMS is a **hand-written, non-differentiable post-processing step**, and that is precisely
what DETR and YOLOv10 set out to remove — hold that thought for Section 3.

In [ ]:
def nms(boxes, scores, iou_thr=0.5):
    keep, idx = [], scores.argsort(descending=True)
    while idx.numel() > 0:
        i = idx[0]
        keep.append(i.item())
        if idx.numel() == 1:
            break
        ious = box_iou(boxes[i][None], boxes[idx[1:]])[0]
        idx = idx[1:][ious <= iou_thr]         # drop everything that overlaps too much
    return torch.tensor(keep, dtype=torch.long)


# Five noisy detections around two true objects.
boxes = torch.tensor([
    [10., 10., 50., 50.], [12., 12., 52., 52.], [ 9.,  8., 48., 49.],
    [80., 80., 120., 120.], [82., 79., 121., 118.],
])
scores = torch.tensor([0.90, 0.85, 0.75, 0.95, 0.60])

kept = nms(boxes, scores, iou_thr=0.5)
print("input boxes :", len(boxes))
print("kept indices:", kept.tolist(), "-> scores", scores[kept].tolist())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, sel, title in [(ax[0], range(len(boxes)), "before NMS (%d boxes)" % len(boxes)),
                      (ax[1], kept.tolist(), "after NMS (%d boxes)" % len(kept))]:
    for i in sel:
        x1, y1, x2, y2 = boxes[i]
        a.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec="tab:red", lw=2))
        a.text(x1, y1 - 2, "%.2f" % scores[i], color="tab:red", fontsize=9)
    a.set_xlim(0, 140); a.set_ylim(0, 140); a.invert_yaxis(); a.set_title(title)
plt.tight_layout(); plt.show()

## 1.4 mAP: the metric everyone quotes and few can define

For one class, at one IoU threshold:

1. Sort all predictions across the dataset by confidence.
2. Walk down the list. A prediction is a **true positive** if it matches an unmatched
   ground-truth box with IoU ≥ threshold; otherwise a **false positive**.
3. After each step compute running precision and recall — this traces the
   precision-recall curve.
4. **AP** is the area under that curve.

**mAP** averages AP over classes. **mAP@[.5:.95]** — the COCO headline number — averages
that again over IoU thresholds from 0.50 to 0.95 in steps of 0.05, which is why COCO mAP
numbers look so much lower than the Pascal VOC mAP@0.5 you may have seen in older papers.

In [ ]:
def average_precision(pred_boxes, pred_scores, gt_boxes, iou_thr=0.5):
    # Single class, single image, for teaching. Real implementations run over a dataset.
    order = pred_scores.argsort(descending=True)
    pred_boxes = pred_boxes[order]
    matched = torch.zeros(len(gt_boxes), dtype=torch.bool)
    tp = torch.zeros(len(pred_boxes))

    for i, pb in enumerate(pred_boxes):
        if len(gt_boxes) == 0:
            break
        ious = box_iou(pb[None], gt_boxes)[0]
        ious[matched] = -1                       # a GT box can only be claimed once
        best = ious.argmax()
        if ious[best] >= iou_thr:
            tp[i] = 1
            matched[best] = True

    tp_cum = tp.cumsum(0)
    fp_cum = (1 - tp).cumsum(0)
    recall = tp_cum / max(len(gt_boxes), 1)
    precision = tp_cum / (tp_cum + fp_cum)

    # 101-point interpolated AP, as COCO does it
    ap, prec_at = 0.0, precision.flip(0).cummax(0).values.flip(0)
    for t in torch.linspace(0, 1, 101):
        p = prec_at[recall >= t]
        ap += (p.max().item() if p.numel() else 0.0) / 101
    return ap, precision, recall


gt = torch.tensor([[10., 10., 50., 50.], [80., 80., 120., 120.], [200., 200., 240., 240.]])
ap, prec, rec = average_precision(boxes, scores, gt, iou_thr=0.5)
print("AP@0.5 = %.4f   (3 ground-truth objects, %d predictions)" % (ap, len(boxes)))

plt.figure(figsize=(5, 4))
plt.step(rec, prec, where="post")
plt.xlabel("recall"); plt.ylabel("precision"); plt.xlim(0, 1); plt.ylim(0, 1.05)
plt.title("Precision-Recall curve (AP = %.3f)" % ap); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

---
# 2. Anchors, and life after them

![anchor boxes](img/anchorbox.jpg)

**Anchor-based** detectors (YOLOv2–v5, Faster R-CNN, RetinaNet) tile the image with a
fixed set of reference boxes at several scales and aspect ratios, then predict an
*offset* from each anchor rather than absolute coordinates. This makes regression easier
— you nudge a box rather than invent one — but it introduces hyperparameters (how many
anchors? what shapes?) that must be tuned per dataset, usually by k-means over the
training boxes.

![multi-scale](img/yolo_Scales.png)

**Multi-scale prediction** is the other half of the classic design: predict from several
feature-map resolutions so that small objects are found in high-resolution early layers
and large objects in coarse late layers. A **Feature Pyramid Network** adds a top-down
path so the high-resolution maps also carry semantic information.

**Anchor-free** detectors (FCOS, YOLOX, YOLOv8 and later) dropped anchors entirely and
regress the box directly from each feature-map location, using a *dynamic label
assignment* rule to decide which locations are responsible for which object. Fewer
hyperparameters, same or better accuracy. This is where modern YOLO sits.

---
# 3. The practical path: YOLO11 with Ultralytics

For real work you do not write a detector from scratch. You take a good pretrained one
and fine-tune it on your data. Ultralytics wraps this into a few lines, and its dataset
format is the de facto standard.

### The YOLO label format

One `.txt` per image, same basename, one line per object:

```
<class_id> <cx> <cy> <w> <h>
```

with `cx cy w h` **normalised to [0, 1]** by image width and height. Directory layout:

```
mydataset/
  images/train/img001.jpg
  images/val/img050.jpg
  labels/train/img001.txt
  labels/val/img050.txt
  data.yaml
```

and `data.yaml`:

```yaml
path: ./mydataset
train: images/train
val: images/val
names:
  0: healthy
  1: diseased
```

In [ ]:
from ultralytics import YOLO

# 'n' = nano, the smallest variant. It fine-tunes comfortably on a free T4.
model = YOLO("yolo11n.pt")
print(model.model.yaml["nc"], "classes in the pretrained head")

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "https://ultralytics.com/images/bus.jpg"
img = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

results = model.predict(img, conf=0.25, verbose=False)
r = results[0]

print("%-14s %-10s %s" % ("class", "conf", "xyxy"))
for box in r.boxes:
    cid = int(box.cls)
    print("%-14s %-10.3f %s" % (r.names[cid], float(box.conf),
                                [round(v, 1) for v in box.xyxy[0].tolist()]))

plt.figure(figsize=(7, 9))
plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off")
plt.title("YOLO11n - pretrained on COCO"); plt.show()

### Fine-tuning on your own data

`coco8` is an 8-image toy dataset bundled with Ultralytics — it downloads in seconds and
lets you verify the training pipeline end to end before you spend an hour on real data.
**Always smoke-test on a tiny dataset first.**

To use your own data, replace `data="coco8.yaml"` with the path to your `data.yaml`.
[Roboflow](https://roboflow.com) is a convenient way to label images and export directly
in this format.

In [ ]:
# Smoke test: 10 epochs on 8 images. Confirms the pipeline runs, not that it learns.
train_results = model.train(
    data="coco8.yaml",
    epochs=10,
    imgsz=640,
    batch=8,
    device=0 if device == "cuda" else "cpu",
    project="runs", name="smoke_test", exist_ok=True,
    verbose=False,
)
print("done - metrics written to runs/smoke_test/")

In [ ]:
metrics = model.val(data="coco8.yaml", verbose=False)
print("mAP@0.5      : %.4f" % metrics.box.map50)
print("mAP@0.5:0.95 : %.4f" % metrics.box.map)
print("mean precision: %.4f   mean recall: %.4f" % (metrics.box.mp, metrics.box.mr))

### Reading a detector's failure modes

When your mAP is disappointing, the number itself tells you nothing. Split it:

| Symptom | Likely cause |
|---|---|
| High precision, low recall | Confidence threshold too high; or objects too small for `imgsz` |
| Low precision, high recall | Threshold too low; or NMS IoU too strict, leaving duplicates |
| Good mAP@0.5, poor mAP@0.5:0.95 | Boxes are found but poorly localised — check label quality |
| One class much worse | Class imbalance, or genuinely ambiguous labels |

Look at `runs/smoke_test/` — Ultralytics writes a confusion matrix, PR curves and
labelled validation batches automatically. Read them before changing anything.

---
# 4. Detection as set prediction: DETR

Everything above has a hand-written component: anchors, label assignment rules, NMS.
[DETR](https://arxiv.org/abs/2005.12872) removed all of it with one idea.

A DETR predicts a **fixed-size set** of N boxes (say 100), always. Training then needs to
decide which prediction should be responsible for which ground-truth object — and it does
so by solving a **bipartite matching problem**: find the one-to-one assignment between
predictions and ground truths that minimises total cost, where cost combines
classification error and box error.

Because the matching is one-to-one, **each object is claimed by exactly one prediction**.
No two predictions are rewarded for the same object, so duplicates never arise and
**NMS is unnecessary**.

The assignment is solved with the Hungarian algorithm. Let's do one by hand.

In [ ]:
from scipy.optimize import linear_sum_assignment

# 5 predictions, 2 ground-truth objects.
pred_boxes = torch.tensor([
    [ 12.,  11.,  49.,  52.],
    [ 78.,  82., 118., 121.],
    [  9.,   9.,  51.,  50.],
    [200., 200., 240., 240.],
    [ 60.,  60.,  90.,  90.],
])
pred_scores = torch.tensor([0.88, 0.91, 0.79, 0.20, 0.15])
gt_boxes = torch.tensor([[10., 10., 50., 50.], [80., 80., 120., 120.]])

# Cost = (negative class confidence) + (negative IoU). DETR also adds an L1 box term.
cost = -pred_scores[:, None] - box_iou(pred_boxes, gt_boxes)
row, col = linear_sum_assignment(cost.numpy())

print("cost matrix (rows = predictions, cols = ground truth):")
print(np.round(cost.numpy(), 3))
print()
for r, c in zip(row, col):
    print("prediction %d  <->  ground truth %d   (IoU %.3f)"
          % (r, c, box_iou(pred_boxes[r][None], gt_boxes[c][None]).item()))
unmatched = sorted(set(range(len(pred_boxes))) - set(row))
print("unmatched predictions -> trained toward the 'no object' class:", unmatched)

Prediction 2 also overlaps ground truth 0 well, but prediction 0 overlaps slightly
better, so the matching gives the object to prediction 0 and pushes prediction 2 toward
"no object". That is how duplicate suppression becomes part of the **loss** instead of a
post-processing step.

### What came next

DETR itself was slow to converge (500 epochs) and weak on small objects. The descendants
fixed that:

- **Deformable DETR** — attend to a few sampled points instead of every pixel. ~10x faster convergence.
- **DINO-DETR** — better query initialisation and denoising training. Became SOTA on COCO.
- **RT-DETR** — an efficient hybrid encoder, making set-prediction detection genuinely
  real-time. First transformer detector to beat YOLO on the speed/accuracy curve.
- **YOLOv10** — brought the NMS-free idea *back into* YOLO with dual label assignment:
  a one-to-many head for rich training signal, a one-to-one head for inference.

Ultralytics ships RT-DETR with the same API, so you can compare directly.

In [ ]:
from ultralytics import RTDETR

rtdetr = RTDETR("rtdetr-l.pt")
res = rtdetr.predict(img, conf=0.25, verbose=False)[0]

print("RT-DETR found %d objects (no NMS involved):" % len(res.boxes))
for box in res.boxes:
    print("  %-14s %.3f" % (res.names[int(box.cls)], float(box.conf)))

fig, ax = plt.subplots(1, 2, figsize=(13, 8))
ax[0].imshow(r.plot()[:, :, ::-1]);   ax[0].set_title("YOLO11n (anchor-free + NMS)"); ax[0].axis("off")
ax[1].imshow(res.plot()[:, :, ::-1]); ax[1].set_title("RT-DETR (set prediction, NMS-free)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

---
# Exercises

**1. NMS threshold sweep.** Using the pretrained YOLO11n on an image with several
overlapping objects (a crowd scene works well), sweep the NMS IoU threshold from 0.1 to
0.9 and plot the number of surviving detections. Then explain the failure mode at each
extreme: what happens to two genuinely overlapping people at IoU 0.1, and what happens at
IoU 0.9?

**2. Implement Soft-NMS.** Instead of deleting overlapping boxes, *decay* their scores by
a factor of exp(−IoU²/σ). Compare against hard NMS on a crowded image. When does it help?

**3. Implement GIoU.** Plain IoU is zero for any two disjoint boxes, so it gives no
gradient when a prediction misses completely.
[Generalised IoU](https://arxiv.org/abs/1902.09630) adds a penalty based on the smallest
enclosing box. Implement `generalized_box_iou` and show it produces a useful signal where
IoU produces none.

**4. Fine-tune on your own data.** Take the dataset from Lab 04 Exercise 5, label at
least 100 images with bounding boxes (Roboflow or LabelImg), and fine-tune YOLO11n.
Report mAP@0.5 and mAP@0.5:0.95, and include the confusion matrix Ultralytics generates.
Then diagnose your worst class using the table in Section 3.

**5. Model size vs. speed.** Benchmark `yolo11n`, `yolo11s` and `yolo11m` on the same 50
images: measure mAP and mean inference latency. Plot accuracy against latency. Which
point on that curve would you ship for a real-time application at 30 FPS, and why?

**6. Set prediction, by hand.** Extend the Hungarian matching cell to include the L1 box
distance term that DETR actually uses, weighted as in the paper (λ_L1 = 5, λ_GIoU = 2).
Does the assignment change? Construct a case where adding the L1 term changes which
prediction wins.